# Building a RAG Pipeline Using PDF Chunking and Retrieval

Loads three public PDF reports on AI governance and digital development, splits them into chunks, embeds the chunks with Azure OpenAI, stores them in a FAISS vector index, and answers questions with a retrieval-augmented LLM chain that returns its source chunks.

See the [README](README.md) for setup and the [project report](report/report.md) for results and lessons learned.

> **Note on outputs:** the outputs shown are from the original run. API credentials have been removed; to re-run, create a `.env` file as described in the README.

## Step 1: Install required packages

In [ ]:
%pip install -U langchain langchain-openai langchain-community langchain-classic langchain-text-splitters faiss-cpu tiktoken pypdf python-dotenv

## Step 2: Import dependencies

In [ ]:
import os
import time
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA

## Step 3: Load Azure OpenAI credentials

Credentials are read from a local `.env` file (never committed to Git). See `.env.example` for the variable names.

In [ ]:
load_dotenv()

required = ["AZURE_OPENAI_API_KEY", "AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_VERSION",
            "AZURE_OPENAI_CHAT_DEPLOYMENT", "AZURE_OPENAI_EMBEDDING_DEPLOYMENT"]
missing = [v for v in required if not os.getenv(v)]
if missing:
    raise EnvironmentError(f"Missing environment variables: {missing}. Copy .env.example to .env and fill them in.")

## Step 4: Load and chunk the PDFs

Place the three PDFs in the `data/` folder (download links are in `data/README.md`).

In [22]:
# 1. Find the three report files anywhere in the project folder
target_files = [
    "nisti_ai_rmf.pdf",
    "un_scientific_panel_ai_preliminary_report.pdf",
    "undp_digital_strategy.pdf"
]

pdf_paths = []
for root, dirs, files in os.walk("."):
    for file in files:
        if file in target_files:
            pdf_paths.append(os.path.join(root, file))

print(f"Found target PDFs at: {pdf_paths}")

# 2. Load the PDFs (one LangChain Document per page, with source file and page number as metadata)
documents = []
for file_path in pdf_paths:
    loader = PyPDFLoader(file_path)
    loaded_pages = loader.load()
    print(f"Loaded {len(loaded_pages)} pages from {file_path}")
    documents.extend(loaded_pages)

# 3. Split into chunks of up to 2000 characters with 200 characters of overlap
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
docs = text_splitter.split_documents(documents)

print(f"\nSuccessfully created {len(docs)} chunks across {len(pdf_paths)} PDFs!")

Found target PDFs at: ['./rag_project/nisti_ai_rmf.pdf', './rag_project/un_scientific_panel_ai_preliminary_report.pdf', './rag_project/undp_digital_strategy.pdf']
Loaded 48 pages from ./rag_project/nisti_ai_rmf.pdf


Loaded 59 pages from ./rag_project/un_scientific_panel_ai_preliminary_report.pdf


Loaded 52 pages from ./rag_project/undp_digital_strategy.pdf

Successfully created 309 chunks across 3 PDFs!


## Step 5: Chunking metrics

Total chunks and average chunk size, as required by the project brief.

> This cell was added after the original run, so it has no saved output. The same chunking was reproduced separately (identical page counts and the same 309 chunks), giving an **average chunk size of about 1,462 characters** (median about 1,905; range 45 to 1,999). See the report for the per-document breakdown.

In [ ]:
chunk_lengths = [len(d.page_content) for d in docs]
print(f"Total chunks: {len(docs)}")
print(f"Average chunk size: {sum(chunk_lengths) / len(chunk_lengths):.0f} characters")
print(f"Smallest / largest chunk: {min(chunk_lengths)} / {max(chunk_lengths)} characters")

## Step 6: Create the embedding model

> This cell was missing from the exported notebook (the `embeddings` object used in Step 7 was defined in a cell that did not survive the export). It has been restored here using the deployment named in the project report.

In [ ]:
embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
    azure_deployment=os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"],
)

## Step 7: Embed the chunks and build the FAISS vector store

The embedding endpoint returned rate-limit errors when all chunks were sent at once, so chunks are embedded in small batches with a pause between batches (about 15 minutes in total for 309 chunks).

In [13]:
def embed_in_batches(docs, embeddings, batch_size=5, pause_seconds=15):
    """Embed documents in small batches to stay under the endpoint's rate limit."""
    vectorstore = None
    for i in range(0, len(docs), batch_size):
        batch = docs[i:i + batch_size]
        if vectorstore is None:
            vectorstore = FAISS.from_documents(batch, embeddings)
        else:
            vectorstore.add_documents(batch)
        print(f"Embedded {min(i + batch_size, len(docs))}/{len(docs)} chunks")
        time.sleep(pause_seconds)
    return vectorstore

vectorstore = embed_in_batches(docs, embeddings)
print(f"Vector store created with {vectorstore.index.ntotal} vectors")

Embedded 5/309 chunks
Embedded 10/309 chunks
Embedded 15/309 chunks
... (progress lines trimmed for readability) ...
Embedded 309/309 chunks
Vector store created with 309 vectors


## Step 8: Initialize the chat model

In [14]:
# gpt-5-mini deployment on Azure OpenAI
llm = AzureChatOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
    deployment_name=os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"],
)

print("Successfully initialized Azure OpenAI LLM!")

Successfully initialized Azure OpenAI LLM!


## Step 9: Create the RAG chain

In [15]:
# RetrievalQA retrieves the most similar chunks and passes them to the LLM as context.
# as_retriever() with no arguments uses the default of 4 chunks per query.
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    return_source_documents=True
)

## Step 10: Ask a question

In [25]:
# Send a question to the RAG chain and store the result

query = "What are the primary findings of the reports on AI?"
result = qa_chain.invoke({"query": query})

## Step 11: Show the answer and the source chunks

In [26]:
# Display the final answer and the source chunks used

print("Answer:", result["result"])
print("\n--- Sources ---")
for i, doc in enumerate(result["source_documents"], 1):
    print(f"\nSource {i}:")
    print(doc.page_content)

Answer: Here are the report’s primary findings, summarized from the Executive Summary, domain findings and the gaps/next-steps sections:

High-level findings
- AI is a rapidly advancing general-purpose technology with large positive potential, but benefits are not automatic — they require complementary investments (data, skills, workflows, organizational redesign).  
- Capabilities are advancing faster than our ability to measure, evaluate and govern them.  
- Only a handful of actors have trained frontier models; development and benefits are concentrated.  
- Inputs and outcomes are geographically and linguistically uneven, and the AI divide is as much about capacity to influence development as about access.  
- For AI to be useful it must be supported by an enabling environment (skills, data governance, institutional adaptation).  
- Emergent “agentic” AI systems represent a governance step change that raises new risks and oversight needs.  
- AI can erode shared reality (misinformat